# 01 — Ingest AyurGenix structured dataset (CSV)

**What this notebook does**
- Reads **`AyurGenixAI_Dataset.csv`** from the UC Volume with Spark **CSV** reader (same path as before).
- **Renames columns** to Unity Catalog–safe identifiers (letters, digits, `_` only). The Kaggle CSV uses characters UC rejects in table columns (`&`, `/`, `(`, `)`, spaces, etc.) — raw `read_files` → `CREATE TABLE` fails with **SQLSTATE 42K05** unless you use column mapping or clean names.
- Writes Delta **`ayurveda_lakehouse.ayurgenix.ayurgenix_dataset`** for Genie / ML / chatbot use.

**Prerequisite:** `00_setup_catalog_schema_volume` completed; CSV is on the Volume.

**Dataset:** [AyurGenixAI: Ayurvedic Dataset](https://www.kaggle.com/datasets/kagglekirti123/ayurgenixai-ayurvedic-dataset)

**How to run** — Serverless → **Run all**.

**Full ingestion:** notebooks **03–04** add **PDF text chunks** as a **secondary** source alongside this **primary** CSV table.


In [0]:
%sql
USE CATALOG ayurveda_lakehouse;
USE SCHEMA ayurgenix;


## Load AyurGenix CSV into Delta (UC-safe column names)

Spark reads the header row, then each original column name is **sanitized** (non-alphanumeric → `_`, leading digits prefixed). Example: `Hindi Name` → `Hindi_Name`, `Diagnosis & Tests` → `Diagnosis_Tests`.

If you need the **original** spellings for documentation, keep a one-row mapping in a spreadsheet or describe columns in your Genie instructions from the CSV file in Git.


In [0]:
import re

CATALOG = "ayurveda_lakehouse"
SCHEMA = "ayurgenix"
TABLE = "ayurgenix_dataset"
VOLUME_CSV = "/Volumes/ayurveda_lakehouse/ayurgenix/source_vault/AyurGenixAI_Dataset.csv"
FQN = f"{CATALOG}.{SCHEMA}.{TABLE}"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

df = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("escape", '"')
    .csv(VOLUME_CSV)
)


def sanitize_column(name: str) -> str:
    s = (name or "").strip()
    s = re.sub(r"[^A-Za-z0-9_]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    if not s:
        s = "col"
    if s[0].isdigit():
        s = "_" + s
    return s


old_cols = list(df.columns)
seen = set()
new_cols = []
for c in old_cols:
    base = sanitize_column(c)
    cand = base
    n = 0
    while cand in seen:
        n += 1
        cand = f"{base}_{n}"
    seen.add(cand)
    new_cols.append(cand)

df_out = df.toDF(*new_cols)
(
    df_out.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FQN)
)

row_count = spark.table(FQN).count()
print(f"Created {FQN}: {len(new_cols)} columns, {row_count} rows")
spark.table(FQN).printSchema()


## Preview
Inspect rows below; column names are **sanitized** for Unity Catalog.


In [0]:
%sql
SELECT * FROM ayurveda_lakehouse.ayurgenix.ayurgenix_dataset LIMIT 10;


**Next step:** open `02_prepare_ayurgenix_curated`.
